# Notebook 01: Data Exploration

**AI Power Electronics Diagnostics — IEEE IES Industrial AI Lab**

This notebook explores the synthetic power electronics signal datasets:
- 3-phase Voltage Source Inverter (VSI) signals with IGBT faults
- PMSM motor drive current signals with motor faults

We examine:
1. Signal characteristics per fault class
2. Class balance
3. Statistical properties (mean, std, kurtosis)
4. Visual differentiation between healthy and faulty signals

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from datasets.synthetic import InverterFaultSimulator, MotorDriveSimulator
from visualization import WaveformPlotter

plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 1. Inverter Fault Signals

Generate one sample per fault class and inspect them.

In [ ]:
sim = InverterFaultSimulator()
fault_types = list(sim.fault_labels().keys())
print(f'Fault classes ({len(fault_types)}): {fault_types}')

In [ ]:
# Generate one sample per fault
signals_dict = {}
for ft in fault_types:
    s, _ = sim.generate(ft)
    signals_dict[ft] = s

plotter = WaveformPlotter(f_sample=100_000.0)
fig = plotter.plot_all_fault_types(signals_dict, channel_idx=3)
plt.show()

In [ ]:
# Healthy vs open-circuit comparison
s_healthy, _ = sim.generate('healthy')
s_fault, _   = sim.generate('open_circuit_T1')

fig = plotter.plot_fault_comparison(s_healthy, s_fault, 'open_circuit_T1', channel_idx=3)
plt.show()

## 2. Generate Full Dataset and Inspect Statistics

In [ ]:
print('Generating inverter dataset...')
X, y = sim.generate_dataset(n_per_class=100, window_size=1024)
print(f'X shape: {X.shape}, y shape: {y.shape}')

label_names = {v: k for k, v in sim.fault_labels().items()}
class_counts = pd.Series(y).map(label_names).value_counts()
print('\nClass distribution:')
print(class_counts)

In [ ]:
# Per-class RMS statistics
stats = []
for label_idx in sorted(label_names):
    mask = y == label_idx
    rms = np.sqrt(np.mean(X[mask] ** 2, axis=(-2, -1)))
    kurt = np.mean([(((X[mask, c] - X[mask, c].mean()) / (X[mask, c].std() + 1e-8)) ** 4).mean()
                    for c in range(X.shape[1])])
    stats.append({'fault': label_names[label_idx], 'mean_rms': rms.mean(), 'std_rms': rms.std(), 'mean_kurtosis': kurt})

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

In [ ]:
# Class-wise RMS box plot
rms_data = []
for label_idx in sorted(label_names):
    mask = y == label_idx
    rms = np.sqrt(np.mean(X[mask, 3] ** 2, axis=-1))  # channel Ia
    for r in rms:
        rms_data.append({'fault': label_names[label_idx], 'rms': r})

df_rms = pd.DataFrame(rms_data)
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=df_rms, x='fault', y='rms', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('Current Ia RMS Distribution per Fault Class')
ax.set_ylabel('RMS [A]')
plt.tight_layout()
plt.show()

## 3. Motor Drive Fault Signals

In [ ]:
motor_sim = MotorDriveSimulator()
motor_faults = list(motor_sim.fault_labels().keys())
print(f'Motor fault classes: {motor_faults}')

motor_plotter = WaveformPlotter(f_sample=50_000.0)
motor_signals = {ft: motor_sim.generate(ft)[0] for ft in motor_faults}
fig = motor_plotter.plot_all_fault_types(motor_signals, channel_idx=0)
plt.show()

In [ ]:
# Bearing fault shows characteristic amplitude modulation
s_bearing, _ = motor_sim.generate('bearing')
s_healthy_m, _ = motor_sim.generate('healthy')

fig = motor_plotter.plot_fault_comparison(s_healthy_m, s_bearing, 'bearing', channel_idx=0)
plt.show()
print('Notebook 01 complete.')